![nn_architecture](media/nn_architecture.png)
![matrix_dims](media/matrix_dims.png)

In [ ]:
import numpy as np    
class NumpyNeuralNetwork():
    def __init__(self, D=728, K=128, C=10, lr=0.002):
        self.D = D # input dimmension
        self.K = K # hidden units first layer
        self.C = C # number of classes

        # first hidden layer
        self.W = np.random.randn(self.K, self.D) * 0.01
        self.b1 = np.random.randn(self.K, 1) * 0.01

        # output head
        self.U = np.random.randn(self.C, self.K) * 0.01
        self.b2 = np.random.randn(self.C, 1) * 0.01

        self.lr = lr


    def forward(self, X):
        """
        forward pass storing values and returning model output
        """
        # first layer
        z = self.W @ X + self.b1
        self.z = z # cache for backprop
        h = self.ReLU(z)
        self.h = h # cache for backprop

        # output layer
        a = self.U @ self.h + self.b2
        a = self.softmax(a)
        self.a = a # cache for backprop
    
        return(a)
    
    def backward(self, X, y, pred):
        """
        backward pass computing and caching gradients
        """
        delta1 = pred - y
        delta2 = (self.U.T @ delta1) * self.heaviside(self.z) 

        self.dU = delta1 @ self.h.T # derivative of U with repect to loss R^CxK
        self.db2 = delta1 # R^Cx1
        self.dW = delta2 @ X.T # R^KxD
        self.db1 = delta2 # R^Kx1
        # if you want to create a saliency map later this can be useful self.dx = self.W.T @ delta2 # R^Dx1

    
    def optimizer_step(self):
        """
        Vanilla optimizer
        """

        self.W = self.W - self.lr * self.dW
        self.b1 = self.b1 - self.lr * self.db1
        
        self.U = self.U - self.lr * self.dU    
        self.b2 = self.b2 - self.lr * self.b2


    def ReLU(self, x):
        return np.maximum(0, x)
    
    def heaviside(self, x):
        return (x > 0).astype(float)    
    
    def softmax(self, a):
        """
        compute softmax values for each element in a
        """
        if isinstance(a, list): a = np.array(a)
        exp_a = np.exp(a - np.max(a)) # minus max to stop overflows
        return exp_a / np.sum(exp_a)

    def cross_entropy(self, y, pred):
        """
        return cross entropy value
        """
        if isinstance(y, list): y = np.array(y)
        if isinstance(pred, list): pred = np.array(pred)
        return - np.sum(np.multiply(y, np.log(pred)))
    


In [ ]:
nn = NumpyNeuralNetwork()

X = np.random.randn(nn.D, 1)
label = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0]).reshape(nn.C, 1)

pred = nn.forward(X)
print(pred)
nn.backward(X, label, pred)

nn.optimizer_step()

[[0.09849357]
 [0.0983649 ]
 [0.09828701]
 [0.10214513]
 [0.10207851]
 [0.10072976]
 [0.09977177]
 [0.0989531 ]
 [0.09920613]
 [0.10197012]]
